# 🎬 From Data to Insight — SOLUTIONS

> This notebook contains full solutions. Try the workshop notebook first!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
print("Libraries loaded ✅")

---
# 🗄️ Section 1 — Data: Raw Facts

### Task 1 — Load the data

In [ ]:
movies  = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')
tags    = pd.read_csv('data/tags.csv')
links   = pd.read_csv('data/links.csv')

print("movies: ", movies.shape)
print("ratings:", ratings.shape)
print("tags:   ", tags.shape)
print("links:  ", links.shape)

### Task 2 — Explore each table

In [ ]:
movies.head()

In [ ]:
ratings.head()

In [ ]:
tags.head()

In [ ]:
ratings.info()

### Task 3 — Check for missing values

In [ ]:
for name, df_tmp in [("movies", movies), ("ratings", ratings), ("tags", tags), ("links", links)]:
    print(f"--- {name} ---")
    print(df_tmp.isnull().sum())
    print()

---
# 🔧 Section 2 — Structured Data: Data Engineering

### Task 5 — Extract year from title

In [ ]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()
movies[['title', 'title_clean', 'year']].head(10)

### Task 6 — Convert timestamps

In [ ]:
ratings['date']  = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year']  = ratings['date'].dt.year
ratings['month'] = ratings['date'].dt.month
ratings[['userId', 'movieId', 'rating', 'date', 'year']].head()

### Task 7 — Split genres

In [ ]:
movies_genres = (
    movies
    .assign(genre=movies['genres'].str.split('|'))
    .explode('genre')
    .query("genre != '(no genres listed)'")
    [['movieId', 'title_clean', 'year', 'genre']]
    .reset_index(drop=True)
)
print(f"Original movies rows: {len(movies)}")
print(f"movies_genres rows:   {len(movies_genres)}")
movies_genres.head(10)

### Task 8 — Join ratings with movies

In [ ]:
df = ratings.merge(movies[['movieId', 'title_clean', 'year', 'genres']], on='movieId')
print(f"Ratings rows:   {len(ratings)}")
print(f"Merged df rows: {len(df)}")
df.head()

### Task 9 — Sanity check

In [ ]:
n_dupes = df.duplicated(subset=['userId', 'movieId']).sum()
print(f"Duplicate (userId, movieId) pairs: {n_dupes}")
print(f"Rating min: {df['rating'].min()}, max: {df['rating'].max()}")
print("\nNulls in merged df:")
print(df.isnull().sum())

---
# 📊 Section 3 — Analytics: Finding Patterns

### Task 10 — Rating distribution

In [ ]:
fig, ax = plt.subplots()
df['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color=sns.color_palette()[0], edgecolor='white')
ax.set_title("Distribution of Movie Ratings")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()

### Task 11 — Most-rated movies

In [ ]:
top_rated_count = (
    df.groupby('title_clean')['rating']
    .count()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 7))
top_rated_count.sort_values().plot(kind='barh', ax=ax)
ax.set_title("Top 20 Most-Rated Movies")
ax.set_xlabel("Number of ratings")
plt.tight_layout()
plt.show()

### Task 12 — Average rating per genre

In [ ]:
df_genres = df.merge(movies_genres[['movieId', 'genre']], on='movieId')

genre_stats = (
    df_genres
    .groupby('genre')['rating']
    .agg(avg_rating='mean', num_ratings='count')
    .reset_index()
    .query("num_ratings >= 500")
    .sort_values("avg_rating", ascending=False)
)
genre_stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=genre_stats, y='genre', x='avg_rating', ax=ax)
ax.set_title("Average Rating by Genre")
ax.set_xlabel("Average rating")
ax.set_xlim(3.0, 4.5)
plt.tight_layout()
plt.show()

### Task 13 — Rating volume over time

In [ ]:
yearly = df.groupby('year')['rating'].count().rename('n_ratings')

fig, ax = plt.subplots()
yearly.plot(ax=ax, marker='o')
ax.set_title("Number of Ratings per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()

### Task 14 — Prolific vs. light raters

In [ ]:
user_counts = df.groupby('userId')['rating'].count().rename('n_ratings')
user_groups = pd.cut(user_counts, bins=[0, 20, 100, np.inf],
                     labels=['Light (≤20)', 'Medium (21–100)', 'Heavy (>100)'])
df_with_group = df.merge(user_groups.rename('user_group'), on='userId')
avg_by_group = df_with_group.groupby('user_group', observed=True)['rating'].mean()
print(avg_by_group)

---
# 💡 Section 4 — Insight: What Does It Mean?

### Task 15 — Hidden gems vs. crowd pleasers

In [ ]:
movie_stats = (
    df.groupby(['movieId', 'title_clean'])['rating']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_rating', 'count': 'n_ratings'})
    .query("n_ratings >= 50")
    .reset_index()
    .merge(movies[['movieId', 'genres']], on='movieId')
)

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(movie_stats['n_ratings'], movie_stats['avg_rating'], alpha=0.3, s=15)

# Quadrant lines
ax.axvline(movie_stats['n_ratings'].median(), color='grey', linestyle='--', alpha=0.5, label='Median ratings')
ax.axhline(movie_stats['avg_rating'].median(), color='grey', linestyle=':', alpha=0.5, label='Median avg rating')

# Annotate standouts
highlights = movie_stats.nlargest(3, 'avg_rating')[['title_clean', 'n_ratings', 'avg_rating']]
for _, row in highlights.iterrows():
    ax.annotate(row['title_clean'], xy=(row['n_ratings'], row['avg_rating']),
                fontsize=8, xytext=(5, 3), textcoords='offset points')

ax.set_title("Movie Popularity vs. Average Rating")
ax.set_xlabel("Number of ratings (popularity)")
ax.set_ylabel("Average rating (quality)")
ax.legend()
plt.tight_layout()
plt.show()

### Task 16 — Genre trends over time

In [ ]:
df_genres['decade'] = (df_genres['year'] // 10) * 10

decade_genre = (
    df_genres
    .groupby(['decade', 'genre'])['rating']
    .count()
    .rename('n_ratings')
    .reset_index()
)

pivot = decade_genre.pivot(index='decade', columns='genre', values='n_ratings').fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

top_genres = pivot_pct.sum().nlargest(8).index
pivot_pct[top_genres].plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab10')
plt.title("Genre Share of Ratings per Decade")
plt.xlabel("Decade")
plt.ylabel("Share of ratings (%)")
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

---
# 🎯 Section 5 — Action: From Insight to Value

### Task 18 — Genre-based recommender

In [ ]:
movie_stats = (
    df.groupby(['movieId', 'title_clean'])['rating']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_rating', 'count': 'n_ratings'})
    .query("n_ratings >= 20")
    .reset_index()
    .merge(movies[['movieId', 'genres']], on='movieId')
)

def recommend(movie_title, n=5):
    matches = movie_stats[movie_stats['title_clean'].str.contains(movie_title, case=False)]
    if matches.empty:
        print(f"No movie found matching '{movie_title}'")
        return
    target = matches.iloc[0]
    target_genres = set(target['genres'].split('|'))
    print(f"Finding movies similar to: {target['title_clean']} ({target['genres']})")
    print("-" * 60)

    def overlap(genres):
        return len(set(genres.split('|')) & target_genres)

    recs = movie_stats.copy()
    recs['overlap'] = recs['genres'].apply(overlap)
    recs = (
        recs[recs['overlap'] > 0]
        [recs['movieId'] != target['movieId']]
        .sort_values(['overlap', 'avg_rating'], ascending=[False, False])
        .head(n)
    )
    return recs[['title_clean', 'genres', 'avg_rating', 'n_ratings', 'overlap']]

recommend("Toy Story")

### Task 19 — Personalised recommender

In [ ]:
def recommend_for_user(user_id, n=10):
    # Movies the user has already rated highly
    seen = df[df['userId'] == user_id]
    liked = seen[seen['rating'] >= 4.0].merge(movies[['movieId', 'genres']], on='movieId')

    if liked.empty:
        print("User has no high-rated movies yet.")
        return

    # Build a genre preference profile (count of liked movies per genre)
    genre_profile = (
        liked.assign(genre=liked['genres'].str.split('|'))
        .explode('genre')
        .groupby('genre').size()
    )

    seen_ids = set(seen['movieId'])

    def preference_score(genres):
        return sum(genre_profile.get(g, 0) for g in genres.split('|'))

    recs = movie_stats.copy()
    recs = recs[~recs['movieId'].isin(seen_ids)]
    recs['pref_score'] = recs['genres'].apply(preference_score)
    recs = recs.sort_values(['pref_score', 'avg_rating'], ascending=[False, False]).head(n)
    return recs[['title_clean', 'genres', 'avg_rating', 'n_ratings', 'pref_score']]

recommend_for_user(user_id=1)